# Finetune Qwen3 with LLaMA Factory

Please use a **free** Tesla T4 Colab GPU to run this!

Project homepage: https://github.com/hiyouga/LLaMA-Factory

## Install Dependencies

In [30]:
%cd /kaggle/working/
import os
os.chdir('/kaggle/working/')

/kaggle/working


In [33]:
!TOKENIZERS_PARALLELISM=true

In [34]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git

fatal: destination path 'LLaMA-Factory' already exists and is not an empty directory.


In [35]:
%cd LLaMA-Factory

/kaggle/working/LLaMA-Factory


In [36]:
!pip install -e .[torch,bitsandbytes]
!pip install -U bitsandbytes
!pip install datasets
!pip install fsspec gcsfs

Obtaining file:///kaggle/working/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
  Created wheel for llamafactory: filename=llamafactory-0.9.5.dev0-py3-none-any.whl size=27018 sha256=9b676507e54ffa9ffecfa750b5350da27ed2dd6611117120c70dcfe9b119c7ae
  Stored in directory: /tmp/pip-ephem-wheel-cache-sub3117v/wheels/20/0c/00/48c6af52334ea56a6fc40a230ab23450ca07cfacb39aa013eb
Successfully built llamafactory
  Attempting uninstall: llamafactory
    Found existing installation: llamafactory 0.9.5.dev0
    Uninstalling llamafactory-0.9.5.dev0:
      Successfully uninstalled llamafactory-0.9.5.dev0


In [37]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

### Check GPU environment

In [38]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: https://medium.com/mlearning-ai/training-yolov4-on-google-colab-316f8fff99c6")
print("cuda OK")

cuda OK


## Update Identity Dataset

In [9]:
import datasets

orig_ds = datasets.load_dataset("SoelMgd/Poker_Dataset", split="train")

Repo card metadata block was not found. Setting CardData to empty.


In [10]:
TEACHER_MODEL = 'qwen3'
STUDENT_MODEL = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
API_KEY = user_secrets.get_secret("IK_KEY")
SYSTEM_PROMPT = "You are playing Texas Hold'em poker no-limit. You will be given the transcription of a game. As an answer, you will output your next action in upper-case and nothing else."

In [11]:
import numpy as np
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import tqdm
from multiprocessing.pool import ThreadPool
import gc

gc.collect()
with torch.no_grad():
    torch.cuda.empty_cache()

pipeline = None
temperature = None

class DASPipelineQwen:
    def __init__(self, openai_api_key, student_model_id="unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"):
        """
        Initialise le pipeline DAS avec un Teacher (API) et un Student (Local 4-bit).
        """
        # 1. Configuration Student (4-bit quantization)
        bnb_config = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, )

        self.student_model_id = student_model_id
        print(f"Chargement du modèle étudiant : {self.student_model_id}...")

        self.tokenizer = AutoTokenizer.from_pretrained(self.student_model_id, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
                self.student_model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
                )
        self.model.eval()

        # 2. Configuration Teacher (OpenAI Compatible API - ex: Infomaniak)
        # Note: Remplacez base_url par l'URL correcte si différent de l'exemple
        self.client = OpenAI(
                api_key=openai_api_key, base_url="https://api.infomaniak.com/2/ai/48/openai/v1"
                )
        self.teacher_model_name = "openai/gpt-oss-120b"

    def get_teacher_data(self, user_prompt, temperature=0.7):
        """
        Génère la réponse du Teacher avec les logprobs.
        """
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ]
        # Note: Assurez-vous que le modèle supporte logprobs=True
        response = self.client.chat.completions.create(
                model=self.teacher_model_name, messages=messages, temperature=temperature, logprobs=True, top_logprobs=1
                )
        content = response.choices[0].message.content
        reasoning = response.choices[0].message.reasoning
        content = f'<reasoning>{reasoning}</reasoning>{content}'
        logprobs_data = response.choices[0].logprobs
        tokens = []
        logprobs = []
        # On vérifie si logprobs est disponible (certaines API compatibles ne le renvoient pas)
        if logprobs_data:
            for token_info in logprobs_data.content:
                tokens.append(token_info.token)
                logprobs.append(token_info.logprob)
        else:
            raise ValueError("L'API Teacher n'a pas renvoyé de logprobs. Vérifiez la compatibilité.")

        # Compute total log probability (sum of logprobs)
        total_logprob = sum(logprobs) if logprobs else 0.0

        # Compute geometric mean of probabilities
        # P_geom = exp(mean(logprobs))
        mean_logprob = np.exp(np.mean(logprobs)) if logprobs else 0.0
        return {
            "content": content, "tokens": tokens, "logprobs": logprobs, "total_logprob": total_logprob,
            "mean_logprob": mean_logprob, "num_tokens": len(tokens)
            }

    def get_student_logprobs(self, prompt: str, response: str) -> dict:
        """
        Calcule les log-probabilités de la réponse (Student) de manière robuste.
        Utilise la méthode de masquage standard (Labels = -100 pour le prompt).
        """
        # 1. Préparer le texte complet (Prompt + Réponse)
        # On utilise le chat template qui gère proprement les balises <|im_start|>, etc.
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response}
            ]
        full_text = self.tokenizer.apply_chat_template(messages, tokenize=False)

        # 2. Tokenizer le tout
        # return_tensors='pt' nous donne directement les tenseurs PyTorch
        inputs = self.tokenizer(full_text, return_tensors="pt").to(self.model.device)
        input_ids = inputs.input_ids

        # 3. Identifier la longueur du Prompt pour le masquage
        # On regénère le prompt SEUL avec l'amorce de réponse (add_generation_prompt=True)
        # Cela inclut "<|im_start|>assistant\n" à la fin, pour s'aligner parfaitement.
        prompt_messages = [{"role": "user", "content": prompt}]
        prompt_text = self.tokenizer.apply_chat_template(
                prompt_messages, tokenize=False, add_generation_prompt=True
                )

        # On tokenise le prompt seul pour avoir sa longueur exacte en tokens
        prompt_tokens = self.tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).input_ids
        response_start_idx = prompt_tokens.shape[1]

        # 4. Créer les Labels (Masking du Prompt)
        # -100 est l'index ignoré par défaut par CrossEntropyLoss de PyTorch
        labels = input_ids.clone()
        # On masque tout ce qui est avant le début de la réponse
        labels[:, :response_start_idx] = -100

        # 5. Calcul "Clean" avec CrossEntropyLoss
        with torch.no_grad():
            outputs = self.model(input_ids)
            logits = outputs.logits

            # Shift des logits et labels pour la prédiction "next token"
            # logits[t] prédit labels[t+1]
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            # reduction='none' nous donne la perte pour chaque token individuel
            loss_fct = torch.nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
            token_losses = loss_fct(shift_logits.transpose(1, 2), shift_labels)

            # La Loss est par définition -log(p), donc log_prob = -loss
            token_logprobs = -token_losses

            # On ne garde que les tokens de la réponse (ceux qui n'étaient pas masqués à -100)
            # Note: shift_labels a été décalé, donc on utilise son masque
            valid_mask = shift_labels != -100
            valid_logprobs = token_logprobs[valid_mask].cpu().numpy()

        # Calcul des statistiques DAS
        total_logprob = np.sum(valid_logprobs)
        mean_logprob = np.exp(np.mean(valid_logprobs)) if len(valid_logprobs) > 0 else 0.0

        return {
            "total_logprob": total_logprob,
            "mean_logprob":  mean_logprob,
            "num_tokens":    len(valid_logprobs),
            "logprobs":      valid_logprobs.tolist()
            }

    def decide_keep_prompt(self, teacher_answer, student_answer):
        teacher_logprob = teacher_answer.get("mean_logprob", 0.0)
        student_logprob = student_answer.get("mean_logprob", 0.0)

        divergence = teacher_logprob - student_logprob
        return divergence >= 0. and teacher_logprob > 0.6

    def run(self, prompt, temperature):
        # print(f"Traitement du prompt : '{prompt}'")

        # 1. Teacher
        teacher_answer = self.get_teacher_data(prompt, temperature)
        if not teacher_answer:
            return

        # print(f"Réponse Teacher reçue ({len(teacher_answer["content"])} chars).")

        # 2. Student & Calculs
        try:
            student_answer = self.get_student_logprobs(prompt, teacher_answer["content"])

            # 3. Décision
            if self.decide_keep_prompt(teacher_answer, student_answer):
                return {
                    'conversations': [
                        {'from': 'system', 'value': SYSTEM_PROMPT},
                        {'from': 'human', 'value': prompt},
                        {'from': 'gpt', 'value': teacher_answer['content']}
                    ]
                }
            else:
                return None

        except Exception as e:
            print(f"Erreur durant le calcul DAS : {e}")
            import traceback
            traceback.print_exc()
            return None

pipeline = DASPipelineQwen(openai_api_key=API_KEY, student_model_id=STUDENT_MODEL)

results_lt = []
results_ht = []
length = min(1000, len(orig_ds))

def worker(i):
    prompt = orig_ds[i]['question']
    result = pipeline.run(prompt, temperature)
    return result
    
temperature = 0.3
with ThreadPool(10) as p:
    results = list(
            tqdm.tqdm(
                p.imap(worker, range(length // 2)),
                total=length // 2
            )
        )
    results_lt = [r for r in results if r is not None]

temperature = 0.9
with ThreadPool(10) as p:
    results = list(
            tqdm.tqdm(
                p.imap(worker, range(length // 2, length)),
                total=length // 2
            )
        )
    results_ht = [r for r in results if r is not None]

pipeline = None
gc.collect()
with torch.no_grad():
    torch.cuda.empty_cache()

Chargement du modèle étudiant : unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

2026-03-01 14:11:55.999322: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772374316.189604      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772374316.240579      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772374316.664907      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772374316.664932      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772374316.664935      55 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

  0%|          | 0/500 [00:01<?, ?it/s]


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid Authentication', 'type': 'authentication_error', 'param': None, 'code': None}}

In [21]:
import json
print('Low temperature samples: '+str(len(results_lt)))
print('High temperature samples: '+str(len(results_ht)))

path = '/kaggle/working/LLaMA-Factory/data/'
with open(path+'/poker_low_temp.json', 'w') as f:
    json.dump(results_lt, f, indent=2)
with open(path+'/poker_high_temp.json', 'w') as f:
    json.dump(results_ht, f, indent=2)

results_lt = None
results_ht = None

Low temperature samples: 500
High temperature samples: 62


In [22]:
import json
dataset_info_path = '/kaggle/working/LLaMA-Factory/data/dataset_info.json'

with open(dataset_info_path, 'r') as f:
    dataset_info = json.load(f)

dataset_info['poker_high_temp'] = {
    'file_name': 'poker_high_temp.json',
    'formatting': 'sharegpt'
}

dataset_info['poker_low_temp'] = {
    'file_name': 'poker_low_temp.json',
    'formatting': 'sharegpt'
}

with open(dataset_info_path, 'w') as f:
    json.dump(dataset_info, f, indent=2)

## Fine-tune model via Command Line

It takes ~30min for training.

In [24]:
!llamafactory-cli train \
    --stage sft \
    --do_train \
    --model_name_or_path unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit \
    --output_dir /kaggle/working/outputs/stage1 \
    --dataset poker_low_temp \
    --template qwen3_nothink \
    --finetuning_type lora \
    --lora_rank 8 \
    --lora_target all \
    --overwrite_output_dir \
    --plot_loss \
    --trust_remote_code \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --learning_rate 1.0e-4 \
    --num_train_epochs 3.0 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.1 \
    --logging_steps 10 \
    --save_steps 500 \
    --cutoff_len 2048 \
    --max_samples 1000 \
    --preprocessing_num_workers 16 \
    --dataloader_num_workers 4 \
    --fp16 \
    --report_to none

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


2026-02-10 14:29:08.204906: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770733748.226535     993 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770733748.233036     993 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770733748.250624     993 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770733748.250659     993 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770733748.250663     993 computation_placer.cc:177] computation placer alr

In [28]:
!llamafactory-cli train \
    --stage sft \
    --do_train \
    --model_name_or_path unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit \
    --dataset poker_high_temp \
    --adapter_name_or_path /kaggle/working/outputs/stage1 \
    --output_dir /kaggle/working/outputs/stage2 \
    --template qwen3_nothink \
    --finetuning_type lora \
    --lora_rank 8 \
    --lora_target all \
    --overwrite_output_dir \
    --plot_loss \
    --trust_remote_code \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --learning_rate 1.0e-4 \
    --num_train_epochs 3.0 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.1 \
    --logging_steps 10 \
    --save_steps 500 \
    --cutoff_len 2048 \
    --max_samples 1000 \
    --preprocessing_num_workers 16 \
    --dataloader_num_workers 4 \
    --fp16 \
    --report_to none

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


2026-02-10 15:31:47.952388: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770737507.977757    1745 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770737507.985208    1745 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770737508.005783    1745 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770737508.005814    1745 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770737508.005818    1745 computation_placer.cc:177] computation placer alr

## Infer the fine-tuned model

In [52]:
import sys
sys.path.append("/kaggle/working/LLaMA-Factory/src")

In [72]:
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc

%cd /kaggle/working/LLaMA-Factory/

args = dict(
  model_name_or_path=STUDENT_MODEL, # use bnb-4bit-quantized,
  adapter_name_or_path="/kaggle/working/outputs/stage1",
  template="qwen3_nothink"
)
chat_model = ChatModel(args)

messages = [{"role": "user", "content": SYSTEM_PROMPT}]
print("Welcome to the CLI application, use `clear` to remove the history, use `exit` to exit the application.")
while True:
  query = input("\nUser: ")
  if query.strip() == "exit":
    break
  if query.strip() == "clear":
    messages = [{"role": "user", "content": SYSTEM_PROMPT}]
    torch_gc()
    print("History has been removed.")
    continue

  messages.append({"role": "user", "content": query})
  print("Assistant: ", end="", flush=True)

  response = ""
  for new_text in chat_model.stream_chat(messages):
    print(new_text, end="", flush=True)
    response += new_text
  print()
  messages.append({"role": "assistant", "content": response})

torch_gc()

/kaggle/working/LLaMA-Factory


[INFO|tokenization_utils_base.py:2095] 2026-03-01 15:20:51,556 >> loading file vocab.json from cache at /root/.cache/huggingface/hub/models--unsloth--Qwen3-4B-Instruct-2507-unsloth-bnb-4bit/snapshots/7744afa8566e264af1a92a806d8d9aae00cc7c78/vocab.json
[INFO|tokenization_utils_base.py:2095] 2026-03-01 15:20:51,557 >> loading file merges.txt from cache at /root/.cache/huggingface/hub/models--unsloth--Qwen3-4B-Instruct-2507-unsloth-bnb-4bit/snapshots/7744afa8566e264af1a92a806d8d9aae00cc7c78/merges.txt
[INFO|tokenization_utils_base.py:2095] 2026-03-01 15:20:51,558 >> loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--unsloth--Qwen3-4B-Instruct-2507-unsloth-bnb-4bit/snapshots/7744afa8566e264af1a92a806d8d9aae00cc7c78/tokenizer.json
[INFO|tokenization_utils_base.py:2095] 2026-03-01 15:20:51,558 >> loading file added_tokens.json from cache at /root/.cache/huggingface/hub/models--unsloth--Qwen3-4B-Instruct-2507-unsloth-bnb-4bit/snapshots/7744afa8566e264af1a92a806d8d9

[INFO|2026-03-01 15:20:53] llamafactory.model.model_utils.quantization:144 >> Loading ?-bit BITSANDBYTES-quantized model.
[INFO|2026-03-01 15:20:53] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.


[INFO|quantization_config.py:508] 2026-03-01 15:20:53,494 >> Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
[INFO|auto.py:242] 2026-03-01 15:20:53,496 >> 
[INFO|modeling_utils.py:1172] 2026-03-01 15:20:53,498 >> loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--unsloth--Qwen3-4B-Instruct-2507-unsloth-bnb-4bit/snapshots/7744afa8566e264af1a92a806d8d9aae00cc7c78/model.safetensors
[INFO|modeling_utils.py:1243] 2026-03-01 15:20:53,504 >> Will use dtype=torch.bfloat16 as defined in model's config object
[INFO|modeling_utils.py:2341] 2026-03-01 15:20:53,504 >> Instantiating Qwen3ForCausalLM model under default dtype torch.bfloat16.
[INFO|configuration_utils.py:986] 2026-03-01 15:20:53,506 >> Generate config GenerationConfig {
  "eos_token_id": 151645,
  "pad_token_id": 151654
}

[INFO|quantizer_bnb_4bit.py:141] 2026-03-01 15:20:53,658 >>

[INFO|2026-03-01 15:20:57] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-03-01 15:20:57] llamafactory.model.adapter:144 >> Loaded adapter(s): /kaggle/working/outputs/stage1
[INFO|2026-03-01 15:20:57] llamafactory.model.loader:144 >> all params: 4,038,983,168
Welcome to the CLI application, use `clear` to remove the history, use `exit` to exit the application.


...


User:  clear


History has been removed.



User:  [TABLE_CONFIGURATION] BTN=P1 SB=P2 0.5BB BB=P3 1BB  [STACKS] P1: 104.1BB P2: 41.5BB [7h Ah] P3: 56.3BB P4: 199.0BB P5: 104.9BB POT=1.5BB  [PREFLOP] P4: FOLD P5: FOLD P1: RAISE 3BB P2:


Assistant: <reasoning>We need to parse the situation. Table configuration: BTN = P1 (button), SB = P2 (small blind), BB = P3 (big blind). Stacks: P1: 104.1BB, P2: 41.5BB, P3: 56.3BB, P4: 199.0BB, P5: 104.9BB. Pot = 1.5BB (presumably antes). Preflop actions: P4 folds, P5 folds, P1 raises 3BB (so raise to 3BB total? Usually raise to 3BB means total bet 3BB, including previous raise? Actually preflop raise to 3BB means total bet 3BB, which is 3BB over big blind? Let's check: big blind is 1BB, small blind 0.5BB. So raise to 3BB means raise to 3BB total, which is 2BB more than big blind. So effective raise amount 2BB. So the raise is 3BB total, meaning the player puts 3BB into the pot (including the big blind? Actually preflop raise: The player must put 3BB into the pot, which is 3BB total. Since they are in the button (P1) and there's no prior action, they raise to 3BB total. So the pot currently: antes: SB 0.5 + BB 1 = 1.5. Then P1 raises to 3BB, so adds 3BB (total 3BB). So pot now 1.5 + 


User:  exit


In [45]:
!zip -r stage1.zip ./stage1

  adding: stage1/ (stored 0%)
  adding: stage1/training_args.bin (deflated 53%)
  adding: stage1/tokenizer_config.json (deflated 90%)
  adding: stage1/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 81%)
  adding: stage1/README.md (deflated 49%)
  adding: stage1/training_loss.png (deflated 8%)
  adding: stage1/adapter_model.safetensors (deflated 7%)
  adding: stage1/vocab.json (deflated 61%)
  adding: stage1/trainer_log.jsonl (deflated 75%)
  adding: stage1/added_tokens.json (deflated 68%)
  adding: stage1/adapter_config.json (deflated 58%)
  adding: stage1/special_tokens_map.json (deflated 69%)
  adding: stage1/chat_template.jinja (deflated 76%)
  adding: stage1/train_results.json (deflated 37%)
  adding: stage1/checkpoint-96/ (stored 0%)
  adding: stage1/checkpoint-96/scaler.pt (deflated 64%)
  adding: stage1/checkpoint-96/training_args.bin (deflated 53%)
  adding: stage1/checkpoint-96/scheduler.pt (deflated 62%)
  adding: stage1/checkpoint-96/tokenizer_config.json (deflated 90%)
  adding: stage1/checkpoint-96/tokenizer.json (deflated 81%)
  adding: stage1/checkpoint-96/README.md (deflated 65%)
  adding: stage1/checkpoint-96/rng_state_1.pth (deflated 27%)
  adding: st

## Merge the LoRA adapter and optionally upload model

NOTE: the Colab free version has merely 12GB RAM, where merging LoRA of a 8B model needs at least 18GB RAM, thus you **cannot** perform it in the free version.

In [ ]:
!hf auth login

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `kaggle temp` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `kaggle temp`


In [89]:
import pathlib

file = pathlib.Path("/kaggle/working/LLaMA-Factory/src/llamafactory/hparams/parser.py")
text = file.read_text()

text = text.replace(
    "json.load(Path(sys.argv[1]).absolute())",
    "json.loads(Path(sys.argv[1]).read_text())"
)

file.write_text(text)

print("Patched parser.py")

Patched parser.py


In [93]:
import json

args = dict(
  model_name_or_path="Qwen/Qwen3-4B-Instruct-2507",
  adapter_name_or_path="/kaggle/working/outputs/stage1",# load the saved LoRA adapters
  template="qwen3_nothink",                                        # same to the one in training
  finetuning_type="lora",                                   # same to the one in training
  export_dir="qwen3_poker",                          # the path to save the merged model
  export_size=2,                                            # the file shard size (in GB) of the merged model
  export_device="cpu",                                      # the device used in export, can be chosen from `cpu` and `auto`
  export_hub_model_id="UP-4303/Qwen3_poker"               # the Hugging Face hub ID to upload model
)

%cd /kaggle/working/LLaMA-Factory/

json.dump(args, open("qwen3_poker.json", "w", encoding="utf-8"), indent=2)

!llamafactory-cli export qwen3_poker.json

/kaggle/working/LLaMA-Factory


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


2026-03-01 15:59:16.304520: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772380756.326045    1202 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772380756.332367    1202 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772380756.349710    1202 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772380756.349740    1202 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772380756.349743    1202 computation_placer.cc:177] computation placer alr